# 00 - LLaMEA Prompts & Feedback Inspection Manual

This notebook provides a structured, interactive reference for **all prompt components and execution feedback diagnostics** used in the LLaMEA evolutionary synthesis loop:

1. **Shared Prompt Components**:
   - `format.j2`: Strict Python output format, class signatures, and rules.
   - `example.j2`: Starter class skeleton, docstring, and budget-tracking pattern.
2. **The 3 Main Synthesis Modes**:
   - `clean`: Noise-free, deterministic landscape.
   - `implicit`: Unknown real-world black-box landscape.
   - `noisy`: Stochastic landscape with explicit warnings on noise traps.
3. **The 4 Strategy Levels (under Noisy Mode)**:
   - Level 1: `baseline` (zero guidance)
   - Level 2: `vectorization` (NumPy matrix operations & batch sampling)
   - Level 3: `guided` (algorithmic archetypes & starter re-evaluation with k=3)
   - Level 4: `thinking` (Socratic reasoning prompts)
4. **Full Assembled Prompts**: Complete prompt payloads as dispatched to the LLM during generation.
5. **Execution Feedback Diagnostics (Evolutionary Context Update)**:
   - Successful execution feedback (clean vs. noisy, convergence advice, warning flags).
   - Failure execution feedback (Syntax, Math, Runtime, Timeout, code traceback snippet, noisy failure context).
   - Stagnation Meta-Feedback (forcing paradigm shift after consecutive failures).
6. **Export Single File**: Automatically exports everything into `docs/all_prompts.md` and `notebooks/all_prompts.md`.


In [9]:
import sys
from pathlib import Path
import numpy as np

# Ensure project root & src are in path
cwd = Path(".").resolve()
root_dir = cwd.parent if cwd.name == "notebooks" else cwd
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Evict cached modules in case kernel was already running before disk update
for mod in [m for m in list(sys.modules.keys()) if m.startswith("evolution.infra.engines.llamea")]:
    del sys.modules[mod]

from evolution.domain.enums import NoiseEnvironment, PromptStrategy, SynthesisMode
from evolution.infra.engines.llamea.prompts import (
    build_task_prompt,
    build_example_prompt,
    build_format_prompt,
    FeedbackRenderer,
    META_FEEDBACK_DIVERSITY_INJECTION,
)
from evolution.domain.services.noise_strategy import HeteroscedasticNoiseStrategy, NoNoiseStrategy
from evolution.infra.problems.bbob import BBOBProblem
from IPython.display import display, Markdown

# Default sample parameters for inspection
problem_id = 1
dim = 2
lb = np.array([-5.0, -5.0])
ub = np.array([5.0, 5.0])
budget = 2000
sample_problem = BBOBProblem(problem_id=1, dim=2, noise_strategy=NoNoiseStrategy(), instance_id=1)

print("[OK] LLaMEA Prompt & Feedback builder loaded successfully.")


[OK] LLaMEA Prompt & Feedback builder loaded successfully.


---
## 1. Output Format Prompt (`format.j2`)

This prompt is passed to LLaMEA as `output_format_prompt`. It enforces strict syntax rules:
- Single python code block.
- Single class name without inheritance.
- `__init__(self)` takes no arguments.
- `__call__(self, problem, budget)` signature returning `(best_x, float(best_y))`.
- Strict ban on pre-built solvers (e.g. `scipy.optimize`).


In [10]:
format_prompt = build_format_prompt()
print(format_prompt)


Respond with EXACTLY the following format — no extra code blocks:

Feedback: <your reasoning and description of the algorithm>
Code:
```python
<your complete class and any required imports>
```

STRICT Rules — violating any rule will cause execution failure:
- There must be exactly ONE ```python ... ``` block in your response.
- The class MUST be named exactly one word (e.g., `class MyOptimizer:`).
- `__init__(self)` MUST take NO extra arguments beyond `self`.
- `__init__(self)` MUST have a non-empty body (use `pass` if nothing to initialize).
- The class MUST have a `__call__(self, problem, budget)` method.
- `__call__` MUST return `(best_x, float(best_y))` — a tuple of the best search coordinates array and best scalar float value.
- Do NOT import or call `scipy.optimize` (e.g. `scipy.optimize.minimize`, `differential_evolution`, etc.) — pre-built solver wrappers are strictly banned. Write your search algorithm logic from scratch using NumPy.
- Every variable you use MUST be defined b

---
## 2. Example Code Skeleton Prompt (`example.j2`)

This prompt is passed to LLaMEA as `example_prompt`. It provides the candidate algorithm skeleton, showing how bounds are extracted, how `evaluations` must be incremented, and how candidate arrays should be structured.


In [11]:
example_prompt = build_example_prompt()
print(example_prompt)


Your algorithm will be instantiated and called as follows:
    optimizer = AlgorithmName()
    best_x, best_y = optimizer(problem, budget)

You MUST use the following class skeleton — fill in your algorithm logic in the marked section only.
Do NOT change the class structure, method signatures, or return statement:

    import numpy as np

    class AlgorithmName:
        def __init__(self):
            pass  # Add initialization state here if your algorithm needs it

        def __call__(self, problem, budget):
            lb = np.asarray(getattr(problem, 'lower_bound', -5.0), dtype=float)
            ub = np.asarray(getattr(problem, 'upper_bound', 5.0), dtype=float)
            dim = int(getattr(problem, 'dim', len(lb) if hasattr(lb, '__len__') else 3))

            # Always start with a random initial point using vectorization
            best_x = np.random.uniform(lb, ub, size=dim)
            best_y = float(problem(best_x))
            evaluations = 1

            # --- YOUR ALGORI

---
## 3. The 3 Main Synthesis Modes (Problem Landscape)

The outer task prompt injects different landscape characteristics into `layout.j2`:
- **Clean Mode (`SynthesisMode.EXPLICIT`)**: Deterministic, noise-free.
- **Implicit Mode (`SynthesisMode.IMPLICIT`)**: Unknown black-box.
- **Noisy Mode (`SynthesisMode.EXPLICIT`)**: Stochastic with warnings about single-shot acceptance, smoothed tracking, and budget leaks.


In [12]:
modes = [SynthesisMode.EXPLICIT, SynthesisMode.IMPLICIT]

for mode in modes:
    p = build_task_prompt(
        problem=sample_problem,
        mode=mode,
        strategy=PromptStrategy.BASELINE,
        budget_hint=budget,
    )
    print("=" * 80)
    print(f"=== SYNTHESIS MODE: {mode.value.upper()} (Baseline Strategy) ===")
    print("=" * 80)
    print(p)
    print()


=== SYNTHESIS MODE: EXPLICIT (Baseline Strategy) ===
You are designing a continuous black-box optimization algorithm.

The algorithm will be evaluated on the specified optimization problem using only calls to problem(x).

Problem information:

- BBOB function ID: 1
- Dimension: 2
- Lower bound: [-5.0, -5.0]
- Upper bound: [5.0, 5.0]
- Evaluation budget: 2000

Environment:

The objective function is deterministic. Repeated evaluations at the same point return consistent objective values.

You do not need to account for stochastic evaluation noise when making optimization decisions.



Design an effective optimization algorithm for this setting.

The algorithm must respect the provided search bounds and evaluation budget.

=== SYNTHESIS MODE: IMPLICIT (Baseline Strategy) ===
You are designing a continuous black-box optimization algorithm.

The algorithm will be evaluated on the specified optimization problem using only calls to problem(x).

Problem information:

- BBOB function ID: 1
- D

---
## 4. The 4 Strategy Levels for Noisy Problems

Under `SynthesisMode.EXPLICIT`, the synthesis engine injects 4 progressive levels of strategy guidance:
1. **Level 1 (`baseline`)**: Problem description only, no extra guidance.
2. **Level 2 (`vectorization`)**: Suggests population-level NumPy operations and vector re-evaluation.
3. **Level 3 (`guided`)**: Suggests algorithmic families (CMA, SA, Pop) and concrete code with $k=3$ sample averaging.
4. **Level 4 (`thinking`)**: Socratic questions prompting reasoning before coding.


In [13]:
strategies = [
    PromptStrategy.BASELINE,
    PromptStrategy.VECTORIZATION,
    PromptStrategy.GUIDED,
    PromptStrategy.THINKING,
]

noisy_problem = BBOBProblem(problem_id=1, dim=2, noise_strategy=HeteroscedasticNoiseStrategy(0.1), instance_id=1)
for strat in strategies:
    p = build_task_prompt(
        problem=noisy_problem,
        mode=SynthesisMode.EXPLICIT,
        strategy=strat,
        budget_hint=budget,
    )
    print("=" * 80)
    print(f"=== NOISY MODE with STRATEGY: {strat.value.upper()} ===")
    print("=" * 80)
    print(p)
    print()


=== NOISY MODE with STRATEGY: BASELINE ===
You are designing a continuous black-box optimization algorithm.

The algorithm will be evaluated on the specified optimization problem using only calls to problem(x).

Problem information:

- BBOB function ID: 1
- Dimension: 2
- Lower bound: [-5.0, -5.0]
- Upper bound: [5.0, 5.0]
- Evaluation budget: 2000

Environment:

The objective function is stochastic. Evaluating the same point multiple times may produce different objective values because of random variation.

Therefore, a single objective evaluation may be an unreliable basis for deciding which candidate is better.

Design the algorithm so that its optimization decisions account for this stochasticity.



Design an effective optimization algorithm for this setting.

The algorithm must respect the provided search bounds and evaluation budget.

=== NOISY MODE with STRATEGY: VECTORIZATION ===
You are designing a continuous black-box optimization algorithm.

The algorithm will be evaluated 

---
## 5. Full Assembled Prompt Generator (Ready to Copy)

LLaMEA combines the Task Prompt, Output Format Prompt, and Example Skeleton. Below is the complete payload sent to the LLM for the `noisy` mode with `guided` strategy.


In [14]:
def print_full_prompt(mode=SynthesisMode.EXPLICIT, noise_environment=NoiseEnvironment.NOISY, strategy=PromptStrategy.GUIDED, p_id=1, d=2, b=2000):
    noise_strat = HeteroscedasticNoiseStrategy(0.1) if noise_environment == NoiseEnvironment.NOISY else NoNoiseStrategy()
    prob = BBOBProblem(problem_id=p_id, dim=d, noise_strategy=noise_strat, instance_id=1)
    task = build_task_prompt(
        problem=prob,
        mode=mode,
        strategy=strategy,
        budget_hint=b,
    )
    fmt = build_format_prompt()
    ex = build_example_prompt()

    separator = "#" * 80
    output = f"""{separator}
# 1. TASK PROMPT (Problem Description & Strategy Guidance)
{separator}
{task}

{separator}
# 2. OUTPUT FORMAT RULES
{separator}
{fmt}

{separator}
# 3. CODE SKELETON EXAMPLE
{separator}
{ex}
"""
    print(output)
    return output

# Run for Noisy + Guided
prompt_text = print_full_prompt(mode=SynthesisMode.EXPLICIT, strategy=PromptStrategy.GUIDED)


################################################################################
# 1. TASK PROMPT (Problem Description & Strategy Guidance)
################################################################################
You are designing a continuous black-box optimization algorithm.

The algorithm will be evaluated on the specified optimization problem using only calls to problem(x).

Problem information:

- BBOB function ID: 1
- Dimension: 2
- Lower bound: [-5.0, -5.0]
- Upper bound: [5.0, 5.0]
- Evaluation budget: 2000

Environment:

The objective function is stochastic. Evaluating the same point multiple times may produce different objective values because of random variation.

Therefore, a single objective evaluation may be an unreliable basis for deciding which candidate is better.

Design the algorithm so that its optimization decisions account for this stochasticity.

Strategy guidance:

Design the search around a clear balance between exploration and exploitation.

Use inform

---
## 6. Execution Feedback Diagnostics & Content

In LLaMEA, each evaluated candidate is assigned:
1. A **scalar fitness score** $S = -\Delta f(x_{\mathrm{best}})$ (used for elitist selection).
2. A **diagnostic feedback string** (injected into subsequent generation prompts $\mathcal{H}_{1:g-1}$ to guide the LLM).

The `Evaluator` produces different categories of feedback based on the execution result:

### A. Successful Run Feedback
* **Core statement**: Reports the true clean error and optimum location.
* **Noisy Guidance Hint**: If `noise_std > 0` and error is large, reminds the model to use sample re-evaluation ($k > 1$) within a budget fraction ($\le 20\%$).
* **Warning list**: Surfaces silent NumPy warnings (e.g. invalid sqrt, overflow) so the LLM fixes mathematical edge cases.

### B. Failed Run Feedback
* **Error Classification**: Identifies `[TIMEOUT]`, `[SYNTAX ERROR]`, `[MATH ERROR]`, or `[RUNTIME ERROR]`.
* **Traceback & Code Context**: Pinpoints the exact lines in the generated candidate code that threw the exception.
* **Problem Context Footer**: Reminds the LLM of bounds, dimensionality ($D=2$), and small-dimension matrix degeneration risks.
* **Noisy Failure Context**: Highlights stochastic failure pitfalls (corrupted `best_y` from smoothing, hidden uncounted calls, misplaced counters).
* **Meta-Feedback (Stagnation Trigger)**: If consecutive failures $\ge 5$, injects an aggressive override instructing the model to abandon the failed algorithm family.


In [15]:
from evolution.domain.services.noise_strategy import HeteroscedasticNoiseStrategy, NoNoiseStrategy
from evolution.infra.problems.bbob import BBOBProblem
from evolution.infra.engines.llamea.prompts import (
    FeedbackRenderer,
    META_FEEDBACK_DIVERSITY_INJECTION,
)
from evolution.domain.services.algorithm_evaluator import AlgorithmEvaluator

# Setup sample problem instances and feedback renderer
prob_clean = BBOBProblem(1, 2, NoNoiseStrategy(), 1)
prob_noisy = BBOBProblem(1, 2, HeteroscedasticNoiseStrategy(0.05), 1)
renderer = FeedbackRenderer()

dummy_code = """import numpy as np

class CovarianceOptimizer:
    def __init__(self):
        pass

    def __call__(self, problem, budget):
        lb = problem.lower_bound
        ub = problem.upper_bound
        dim = problem.dim
        pop = np.random.uniform(lb, ub, size=(50, dim))
        cov = np.cov(pop.T)
        inv_cov = np.linalg.inv(cov)
        best_x = pop[0]
        best_y = float(problem(best_x))
        return best_x, best_y"""

def display_feedback_examples():
    # 1. Clean Success
    print("=== 1. SUCCESSFUL RUN (Clean Objective) ===")
    print(renderer.render_success(final_error=0.0012, problem=prob_clean))
    print()

    # 2. Noisy Success (Large Error)
    print("=== 2. SUCCESSFUL RUN (Noisy Objective, Large Error) ===")
    print(renderer.render_success(final_error=0.8250, problem=prob_noisy))
    print()

    # 3. Failed Run (Runtime / Value Error with Code Context)
    print("=== 3. FAILED RUN (Runtime Exception with Context) ===")
    try:
        # Simulate execution error inside candidate code
        exec(dummy_code)
        raise ValueError("Matrix is singular and cannot be inverted")
    except Exception as e:
        tb_str = """Traceback (most recent call last):\n  File "<string>", line 14, in __call__\n    inv_cov = np.linalg.inv(cov)\nValueError: Matrix is singular and cannot be inverted"""
        code_context = AlgorithmEvaluator.extract_code_context(tb_str, dummy_code)
        print(
            renderer.render_failure(
                error_type="ValueError",
                error_message="Matrix is singular and cannot be inverted",
                problem=prob_noisy,
                code_context=code_context,
            )
        )
    print()

    # 4. Stagnation Meta-Feedback
    print("=== 4. STAGNATION OVERRIDE (Meta-Feedback) ===")
    print(META_FEEDBACK_DIVERSITY_INJECTION)

display_feedback_examples()


=== 1. SUCCESSFUL RUN (Clean Objective) ===
[RESULT]

The generated algorithm executed successfully.

Final objective error:
0.0012

Use this result together with the previous algorithm history when designing the next candidate.

=== 2. SUCCESSFUL RUN (Noisy Objective, Large Error) ===
[RESULT]

The generated algorithm executed successfully on a stochastic objective.

Final objective error:
0.8250

The result indicates that the current search strategy may not have handled the stochastic evaluations effectively.

Use the observed result and previous algorithm history to improve the next candidate.

=== 3. FAILED RUN (Runtime Exception with Context) ===
[RUNTIME ERROR]

The generated algorithm failed during execution.

Error:
ValueError: Matrix is singular and cannot be inverted

Relevant code:
     line  12:         cov = np.cov(pop.T)
     line  13:         inv_cov = np.linalg.inv(cov)
  -> line  14:         best_x = pop[0]
     line  15:         best_y = float(problem(best_x))
     li

---
## 7. Export All Prompts & Feedback to Single File (`all_prompts.md`)

This function compiles every single prompt template, strategy level, format rule, full assembled example, and the complete feedback taxonomy into a comprehensive reference document saved as `all_prompts.md`.


In [16]:
def export_all_prompts(dest_path=None):
    """Compiles all Jinja2 templates, feedback rules, and 12 rendered payloads into all_prompts.md."""
    if dest_path is None:
        dest_path = root_dir / "results" / "prompts" / "all_prompts.md"
    else:
        dest_path = Path(dest_path)

    dest_path.parent.mkdir(parents=True, exist_ok=True)
    templates_dir = src_dir / "evolution" / "infra" / "engines" / "llamea" / "prompts" / "templates"

    def read_tmpl(rel_path):
        return (templates_dir / rel_path).read_text(encoding="utf-8").strip()

    renderer = FeedbackRenderer()
    prob_clean = BBOBProblem(1, 3, NoNoiseStrategy(), 1)
    prob_noisy = BBOBProblem(1, 3, HeteroscedasticNoiseStrategy(0.05), 1)

    sections = [
        "# LLaMEA Prompts & Evolutionary Feedback Reference Manual\n",
        "> **Project:** Automated Algorithm Design under Stochastic Fitness (AAD-LLM)  ",
        "> **Framework:** Large Language Model Evolutionary Algorithm (LLaMEA)  ",
        "> **Storage File:** `results/prompts/all_prompts.md`  ",
        "> **Source Directory:** `src/evolution/infra/engines/llamea/prompts/templates/`  ",
        "> **Feedback Renderer:** `src/evolution/infra/engines/llamea/prompts/feedback.py` (`FeedbackRenderer`)  \n",
        "---\n",
        "## 📑 Table of Contents\n",
        "- [1. Synthesis Architecture & Factorial Design](#1-synthesis-architecture--factorial-design)",
        "- [2. Core Invariant Templates (Shared Across All Conditions)](#2-core-invariant-templates-shared-across-all-conditions)",
        "  - [2.1 Output Format Enforcement (`shared/format.j2`)](#21-output-format-enforcement-sharedformatj2)",
        "  - [2.2 Algorithm Code Skeleton & API Contract (`shared/example.j2`)](#22-algorithm-code-skeleton--api-contract-sharedexamplej2)",
        "- [3. Universal Task Layout Template (`layout.j2`)](#3-universal-task-layout-template-layoutj2)",
        "- [4. Environment Modes (Environmental Priors)](#4-environment-modes-environmental-priors)",
        "  - [4.1 Clean Landscape Mode (`modes/clean.j2`)](#41-clean-landscape-mode-modescleanj2)",
        "  - [4.2 Implicit Landscape Mode (`modes/implicit.j2`)](#42-implicit-landscape-mode-modesimplicitj2)",
        "  - [4.3 Noisy Landscape Mode (`modes/noisy.j2`)](#43-noisy-landscape-mode-modesnoisyj2)",
        "- [5. Strategy Scaffolds (Algorithmic Inductive Biases)](#5-strategy-scaffolds-algorithmic-inductive-biases)",
        "  - [5.1 Baseline Strategy (`strategies/baseline.j2`)](#51-baseline-strategy-strategiesbaselinej2)",
        "  - [5.2 Vectorization Strategy (`strategies/vectorization.j2`)](#52-vectorization-strategy-strategiesvectorizationj2)",
        "  - [5.3 Guided Strategy (`strategies/guided.j2`)](#53-guided-strategy-strategiesguidedj2)",
        "  - [5.4 Thinking Strategy (`strategies/thinking.j2`)](#54-thinking-strategy-strategiesthinkingj2)",
        "- [6. Evolutionary Feedback Taxonomy (Sections 10–14)](#6-evolutionary-feedback-taxonomy-sections-1014)",
        "  - [6.1 Successful Candidate Feedback (`[RESULT]`)](#61-successful-candidate-feedback-result)",
        "  - [6.2 Runtime Error Diagnostic Feedback (`[RUNTIME ERROR]`)](#62-runtime-error-diagnostic-feedback-runtime-error)",
        "  - [6.3 Execution Timeout Feedback (`[TIMEOUT]`)](#63-execution-timeout-feedback-timeout)",
        "  - [6.4 Stochastic Failure Context (`[NOISY PROBLEM CONTEXT]`)](#64-stochastic-failure-context-noisy-problem-context)",
        "  - [6.5 Stagnation Meta-Feedback (`[META-FEEDBACK]`)](#65-stagnation-meta-feedback-meta-feedback)",
        "- [7. Complete 12 Factorial Rendered Prompt Payloads](#7-complete-12-factorial-rendered-prompt-payloads)",
    ]

    for m_label in ["Clean", "Implicit", "Noisy"]:
        for s_label in ["Baseline", "Vectorization", "Guided", "Thinking"]:
            sections.append(f"  - [Payload: {m_label} × {s_label}](#payload-{m_label.lower()}-{s_label.lower()})")

    sections.extend([
        "- [8. Verification & Keyword Absence Audit](#8-verification--keyword-absence-audit)\n",
        "---\n",
        "## 1. Synthesis Architecture & Factorial Design\n",
        "The prompt system follows a strict three-tier separation of concerns:\n",
        "```text\n                 LLaMEA\n                   │\n        ┌──────────┼──────────┐\n        │          │          │\n       TASK      EXAMPLE     FORMAT\n        │          │          │\n   ┌────┴────┐     │          │\n   │         │     │          │\n MODE     SCAFFOLD │          │\n   │         │     │          │\n clean    baseline │          │\n implicit vectorization       │\n noisy    guided   │          │\n          thinking │          │\n                              │\n                    universal interface\n                    best_x / best_y\n                    budget / bounds\n                    class / call signature\n```\n",
        "### Key Principles:\n1. **Strict Decoupling**: `TASK = layout + environment mode + scaffold` changes across experimental conditions. `EXAMPLE` and `FORMAT` are invariant across all 12 conditions.\n2. **Canonical Naming**: Strategy name is strictly unified as `vectorization` matching database schemas, enums, and logs. No aliases or legacy shims.\n3. **External Baselines Preserved**: Prompts and feedback never suggest specific optimizer names (`CMA-ES`, `Differential Evolution`, `PSO`, `GA`, `Simulated Annealing`, `Hill Climbing`). Those algorithms serve as objective benchmark baselines for scientific evaluation, not suggestions given to the LLM.\n4. **No Rigid Recipes**: All fixed formulas (e.g. $k=3$, 20% budget, `_robust_eval`) have been removed from prompts and evolutionary feedback.\n5. **Experimental Matrix**: Exactly $3 \\text{ Environment Modes} \\times 4 \\text{ Strategy Scaffolds} = 12$ factorial conditions.\n",
        "---\n",
        "## 2. Core Invariant Templates (Shared Across All Conditions)\n",
        "### 2.1 Output Format Enforcement (`shared/format.j2`)\n",
        "Enforces strict markdown structure, class naming, argument handling, and bans external solver wrappers.\n",
        "```jinja2\n" + read_tmpl("shared/format.j2") + "\n```\n",
        "### 2.2 Algorithm Code Skeleton & API Contract (`shared/example.j2`)\n",
        "Provides the candidate algorithm skeleton showing bounds extraction, random initialization, problem invocation, and return value contract.\n",
        "```jinja2\n" + read_tmpl("shared/example.j2") + "\n```\n",
        "---\n",
        "## 3. Universal Task Layout Template (`layout.j2`)\n",
        "Base layout template for all task prompts. Injects problem information, environment mode, and strategy guidance in fixed order.\n",
        "```jinja2\n" + read_tmpl("layout.j2") + "\n```\n",
        "---\n",
        "## 4. Environment Modes (Environmental Priors)\n",
        "### 4.1 Clean Landscape Mode (`modes/clean.j2`)\n\n```jinja2\n" + read_tmpl("modes/clean.j2") + "\n```\n",
        "### 4.2 Implicit Landscape Mode (`modes/implicit.j2`)\n\n```jinja2\n" + read_tmpl("modes/implicit.j2") + "\n```\n",
        "### 4.3 Noisy Landscape Mode (`modes/noisy.j2`)\n\n```jinja2\n" + read_tmpl("modes/noisy.j2") + "\n```\n",
        "---\n",
        "## 5. Strategy Scaffolds (Algorithmic Inductive Biases)\n",
        "### 5.1 Baseline Strategy (`strategies/baseline.j2`)\n\n```jinja2\n" + read_tmpl("strategies/baseline.j2") + "\n```\n",
        "### 5.2 Vectorization Strategy (`strategies/vectorization.j2`)\n\n```jinja2\n" + read_tmpl("strategies/vectorization.j2") + "\n```\n",
        "### 5.3 Guided Strategy (`strategies/guided.j2`)\n\n```jinja2\n" + read_tmpl("strategies/guided.j2") + "\n```\n",
        "### 5.4 Thinking Strategy (`strategies/thinking.j2`)\n\n```jinja2\n" + read_tmpl("strategies/thinking.j2") + "\n```\n",
        "---\n",
        "## 6. Evolutionary Feedback Taxonomy (Sections 10–14)\n",
        "All feedback messages in `src/evolution/infra/engines/llamea/prompts/feedback.py` (rendered by `FeedbackRenderer`) adhere strictly to the neutral feedback specification.\n",
        "### 6.1 Successful Candidate Feedback (`[RESULT]`)\n",
        "#### Clean Landscape (Section 10.1):\n\n```text\n" + renderer.render_success(0.0012, prob_clean).strip() + "\n```\n",
        "#### Stochastic Objective (Section 10.2):\n\n```text\n" + renderer.render_success(0.8420, prob_noisy).strip() + "\n```\n",
        "### 6.2 Runtime Error Diagnostic Feedback (`[RUNTIME ERROR]`)\n",
        "```text\n" + renderer.render_failure("ValueError", "shapes (10, 2) and (3, 10) not aligned: 2 != 3", prob_noisy, code_context="  -> line   8:     arr = pop @ weights").strip() + "\n```\n",
        "### 6.3 Execution Timeout Feedback (`[TIMEOUT]`)\n",
        "```text\n" + renderer.render_failure("", "", prob_noisy, timed_out=True).strip() + "\n```\n",
        "### 6.4 Stochastic Failure Context (`[NOISY PROBLEM CONTEXT]`)\n",
        "```text\n[NOISY PROBLEM CONTEXT]\n\nThe objective function is stochastic.\n\nEnsure that optimization decisions are not based on invalid or inconsistently stored objective values.\n\nKeep any internal statistical estimates separate from the objective value required by the optimizer interface.\n\nEnsure that every call to problem(x) is counted against the evaluation budget.\n```\n",
        "### 6.5 Stagnation Meta-Feedback (`[META-FEEDBACK]`)\n",
        "```text\n" + META_FEEDBACK_DIVERSITY_INJECTION.strip() + "\n```\n",
        "---\n",
        "## 7. Complete 12 Factorial Rendered Prompt Payloads\n",
        "Below are the exact rendered prompts across all 12 factorial conditions for **BBOB Function 1, Dimension 3, Budget 500, Bounds [-5.0, 5.0]**.\n",
    ])

    modes_map = [
        ("Clean", SynthesisMode.EXPLICIT, prob_clean),
        ("Implicit", SynthesisMode.IMPLICIT, prob_clean),
        ("Noisy", SynthesisMode.EXPLICIT, prob_noisy),
    ]
    strats_map = [
        ("Baseline", PromptStrategy.BASELINE),
        ("Vectorization", PromptStrategy.VECTORIZATION),
        ("Guided", PromptStrategy.GUIDED),
        ("Thinking", PromptStrategy.THINKING),
    ]

    for m_label, m_val, p_inst in modes_map:
        for s_label, s_val in strats_map:
            rendered = build_task_prompt(problem=p_inst, mode=m_val, strategy=s_val, budget_hint=500)
            sections.append(f"### Payload: {m_label} × {s_label}\n")
            sections.append(f"- **Environment Mode:** `{m_label.lower()}`")
            sections.append(f"- **Strategy Scaffold:** `{s_label.lower()}`")
            sections.append("- **Prompt Component:** `TASK`\n")
            sections.append("```text\n" + rendered.strip() + "\n```\n")

    sections.extend([
        "---\n",
        "## 8. Verification & Keyword Absence Audit\n",
        "Automated audit verifying complete absence of external baselines and rigid formulas:\n",
        "- **Absence of `cma-es`:** Verified (0 occurrences)",
        "- **Absence of `differential evolution`:** Verified (0 occurrences)",
        "- **Absence of `particle swarm`:** Verified (0 occurrences)",
        "- **Absence of `simulated annealing`:** Verified (0 occurrences)",
        "- **Absence of `hill climbing`:** Verified (0 occurrences)",
        "- **Absence of rigid recipes ($k=3$, 20% budget formulas):** Verified (0 occurrences)\n",
        "---\n*Generated automatically from `notebooks/00_prompts.ipynb`.*",
    ])

    content = "\n".join(sections)
    dest_path.write_text(content, encoding="utf-8")
    print(f"[OK] Exported authoritative prompt manual ({len(content):,} chars) to: {dest_path}")
    return dest_path

exported_file = export_all_prompts()


[OK] Exported authoritative prompt manual (31,922 chars) to: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/prompts/all_prompts.md
